# Caro ResNet


In [ ]:
import copy
import torch
from torch import nn


In [ ]:
BOARD_SIZE = 15
INPUT_CHANNELS = 3

In [ ]:
# Cấu hình dễ tinh chỉnh
MODEL_CHANNELS = 64
MODEL_BLOCKS = 10
LEARNING_RATE = 1e-3

TRAIN_STEPS = 1000
TRAIN_MAX_MOVES = 100
PRINT_EVERY = 1

# Batch training: mỗi slot trong batch là một ván riêng.
# Ván nào kết thúc sớm sẽ được đánh dấu done và đứng yên đến hết rollout.
TRAIN_BATCH_SIZE = 8

FUTURE_STEPS = 10
GAMMA = 0.8
FUTURE_WEIGHT = 1.0

REWARD_ALPHA = 8.0
WIN_REWARD = None

VISUAL_MAX_MOVES = 120
VISUAL_PAUSE = 0.25


## ResidualBlock


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        
        x = x + residual
        x = self.relu(x)
        return x


## Model chính


In [ ]:
class CaroResNet(nn.Module):
    def __init__(self, input_channels=3, channels=32, num_blocks=20):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(input_channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            *[ResidualBlock(channels) for _ in range(num_blocks)]
        )
        self.policy_head = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, board):
        x = self.stem(board)
        x = self.blocks(x)
        x = self.policy_head(x)
        return x.squeeze(1)


## Input mẫu 3 kênh


In [ ]:
# channel 0: our stones
# channel 1: enemy stones
# channel 2: legal cells
board = torch.zeros(1, INPUT_CHANNELS, BOARD_SIZE, BOARD_SIZE)
board[:, 2, :, :] = 1

board[0, 0, 7, 7] = 1
board[0, 2, 7, 7] = 0

board[0, 1, 7, 8] = 1
board[0, 2, 7, 8] = 0

board.shape


In [ ]:
model = CaroResNet(
    input_channels=INPUT_CHANNELS,
    channels=MODEL_CHANNELS,
    num_blocks=MODEL_BLOCKS,
)

scores = model(board)
probabilities = torch.softmax(scores.flatten(start_dim=1), dim=1).reshape(-1, BOARD_SIZE, BOARD_SIZE)

print("Input shape :", board.shape)
print("Scores shape:", scores.shape)
print("Prob shape  :", probabilities.shape)
print("Prob sum   :", probabilities.sum(dim=(1, 2)))
print("số tham số của mô hình:", sum(p.numel() for p in model.parameters()))


## Loss


In [ ]:
def window_reward(board_t, row, col, alpha=REWARD_ALPHA, win_reward=WIN_REWARD):
    """
    Tính điểm thưởng heuristic cho một ô nếu ta giả sử sẽ đánh vào ô đó.

    board_t có shape (3, 15, 15):
        board_t[0]: quân của người sắp đi.
        board_t[1]: quân của đối thủ.
        board_t[2]: ô còn hợp lệ để đánh.

    Hàm này chỉ tính điểm thế cờ cục bộ trong các cửa sổ 5 ô đi qua (row, col).
    Điểm càng cao nghĩa là ô đó càng đáng được policy ưu tiên.
    """
    if win_reward is None:
        win_reward = alpha ** 6

    our = board_t[0]
    enemy = board_t[1]
    legal = board_t[2]
    board_size = board_t.shape[-1]

    # Bốn hướng tạo chuỗi trong caro: dọc, ngang, chéo xuống, chéo lên.
    directions = [(1, 0), (0, 1), (1, 1), (1, -1)]

    total_reward = torch.zeros((), device=board_t.device)

    for dr, dc in directions:
        # Một ô có thể nằm ở 5 vị trí khác nhau trong một cửa sổ dài 5.
        for offset in range(5):
            start_row = row - offset * dr
            start_col = col - offset * dc

            cells = []
            valid_window = True

            for k in range(5):
                r = start_row + k * dr
                c = start_col + k * dc

                if not (0 <= r < board_size and 0 <= c < board_size):
                    valid_window = False
                    break

                cells.append((r, c))

            if not valid_window:
                continue

            our_count = 0
            enemy_count = 0
            has_wall = False

            for r, c in cells:
                # Ô đang xét là ô ta định đánh, nên coi như sẽ được lấp bởi quân ta.
                if r == row and c == col:
                    continue

                if our[r, c] > 0.5:
                    our_count += 1
                elif enemy[r, c] > 0.5:
                    enemy_count += 1
                elif legal[r, c] <= 0.5:
                    # Trạng thái này không nên xuất hiện trong caro thường, nhưng nếu có thì bỏ cửa sổ.
                    has_wall = True

            if has_wall:
                continue

            # Cửa sổ có cả hai phe thì không còn là chuỗi mở có ích cho bên nào.
            if our_count > 0 and enemy_count > 0:
                continue

            if our_count == 4:
                # Đánh vào ô này tạo 5 quân liên tiếp, nên cho reward thắng rất lớn.
                total_reward = total_reward + win_reward
            elif enemy_count == 4:
                # Chặn đối thủ đang có 4 quân cũng phải rất khẩn cấp, gần như ngang nước thắng.
                total_reward = total_reward + win_reward * 0.9
            elif our_count > 0:
                total_reward = total_reward + (alpha ** our_count)
            elif enemy_count > 0:
                # Thưởng phòng thủ thấp hơn tấn công cùng độ dài, trừ trường hợp enemy_count == 4 ở trên.
                total_reward = total_reward + 0.8 * (alpha ** enemy_count)

    return total_reward


def build_reward_map(board_t, alpha=REWARD_ALPHA, win_reward=WIN_REWARD):
    """
    Tạo bản đồ reward cho toàn bộ bàn cờ.

    Chỉ ô hợp lệ mới cần reward thật vì local_step_loss sẽ mask toàn bộ ô không hợp lệ.
    Ô không hợp lệ giữ reward 0 vì luật hợp lệ đã được xử lý bằng legal mask.
    """
    if win_reward is None:
        win_reward = alpha ** 6

    board_size = board_t.shape[-1]
    reward_map = torch.zeros(board_size, board_size, device=board_t.device)

    for row in range(board_size):
        for col in range(board_size):
            if board_t[2, row, col] > 0.5:
                reward_map[row, col] = window_reward(board_t, row, col, alpha=alpha, win_reward=win_reward)

    return reward_map


def normalize_reward_map(reward_map, legal_mask, eps=1e-6):
    """
    Chuẩn hóa reward để tránh gradient quá lớn.

    Chỉ các ô hợp lệ được dùng để tính mean/std. Ô không hợp lệ sẽ bị mask khỏi
    target_policy và log_policy trong local_step_loss, nên giá trị của chúng không tham gia loss.
    """
    normalized = reward_map.clone()
    legal_rewards = reward_map[legal_mask]

    if legal_rewards.numel() > 1:
        mean = legal_rewards.mean()
        std = legal_rewards.std(unbiased=False).clamp_min(eps)
        normalized[legal_mask] = (legal_rewards - mean) / std
    elif legal_rewards.numel() == 1:
        normalized[legal_mask] = 1.0

    normalized[~legal_mask] = 0.0
    return normalized.clamp(min=-5.0, max=5.0)


def local_step_loss(scores_t, board_t, alpha=REWARD_ALPHA, win_reward=WIN_REWARD):
    """
    Tính loss cục bộ cho một bước đi.

    Legal mask là luật cứng của môi trường: model chỉ học phân phối giữa các ô hợp lệ.
    Ô không hợp lệ luôn bị mask khi chọn nước và khi tính loss.
    """
    legal_mask = board_t[2] > 0.5
    reward_map = build_reward_map(board_t, alpha=alpha, win_reward=win_reward)
    normalized_reward = normalize_reward_map(reward_map, legal_mask)

    # Mask score trước softmax để loss chỉ phân bổ xác suất trên các ô hợp lệ.
    masked_scores = scores_t.masked_fill(~legal_mask, -1e9)

    # Target distribution được tạo từ reward đã normalize: reward cao thì target cao.
    target_policy = torch.softmax(normalized_reward.masked_fill(~legal_mask, -1e9).flatten(), dim=0)
    log_policy = torch.log_softmax(masked_scores.flatten(), dim=0)

    return -(target_policy * log_policy).sum()


def compute_local_losses(scores, boards):
    """
    Tính local_step_loss cho danh sách board/scores của learner.
    """
    local_losses = []
    for t in range(scores.shape[0]):
        loss_t = local_step_loss(scores[t], boards[t])
        local_losses.append(loss_t)
    return torch.stack(local_losses)


In [ ]:
def has_five_in_a_row(stones, row, col):
    """
    Kiểm tra sau khi đặt quân tại (row, col), bên vừa đi có tạo được 5 liên tiếp không.

    stones là tensor (15, 15) của riêng phe vừa đi, giá trị 1 nghĩa là có quân.
    """
    board_size = stones.shape[-1]
    directions = [(1, 0), (0, 1), (1, 1), (1, -1)]

    for dr, dc in directions:
        count = 1

        r, c = row + dr, col + dc
        while 0 <= r < board_size and 0 <= c < board_size and stones[r, c] > 0.5:
            count += 1
            r += dr
            c += dc

        r, c = row - dr, col - dc
        while 0 <= r < board_size and 0 <= c < board_size and stones[r, c] > 0.5:
            count += 1
            r -= dr
            c -= dc

        if count >= 5:
            return True

    return False


def select_legal_move(scores_t, legal_mask):
    """
    Chọn nước có score cao nhất nhưng chỉ trong các ô hợp lệ.

    Model vẫn output score cho toàn bộ 15x15 ô. Nếu dùng argmax trực tiếp,
    model có thể chọn lại ô đã có quân. Vì vậy ta mask ô không hợp lệ thành
    số rất âm trước khi argmax để rollout luôn đi nước hợp lệ khi còn ô trống.
    """
    if not legal_mask.any():
        return None

    masked_scores = scores_t.masked_fill(~legal_mask, -1e9)
    move_idx = torch.argmax(masked_scores).item()
    board_size = scores_t.shape[-1]
    row = move_idx // board_size
    col = move_idx % board_size
    return row, col


def step_board_from_model_output(board_t, scores_t):
    """
    Sinh trạng thái bàn kế tiếp từ góc nhìn người sắp đi.

    Trả về:
      (board_next, "continue") nếu nước đi hợp lệ và game chưa kết thúc.
      (board_after, "win") nếu nước đi tạo thắng.
      (board_after, "draw") nếu bàn cờ hết ô hợp lệ sau nước đi.
      (board_t, "invalid") chỉ dùng như fallback nếu trạng thái đầu vào đã bất thường.

    Với trạng thái continue, board_next được đảo kênh để người kế tiếp lại nhìn
    board_t[0] là quân của mình và board_t[1] là quân đối thủ.
    """
    legal_mask = board_t[2] > 0.5
    selected_move = select_legal_move(scores_t, legal_mask)

    if selected_move is None:
        return board_t, "invalid"

    row, col = selected_move

    our_after = board_t[0].clone()
    enemy_after = board_t[1].clone()
    legal_after = board_t[2].clone()

    our_after[row, col] = 1
    legal_after[row, col] = 0

    # board_after_current_view giữ nguyên góc nhìn của người vừa đi.
    # Dùng nó cho terminal states để visualization thấy được nước cuối cùng.
    board_after_current_view = torch.stack([our_after, enemy_after, legal_after], dim=0)

    if has_five_in_a_row(our_after, row, col):
        return board_after_current_view, "win"

    if legal_after.sum() <= 0.5:
        return board_after_current_view, "draw"

    # Nếu game tiếp tục, đổi góc nhìn sang người kế tiếp.
    board_next = torch.stack([enemy_after, our_after, legal_after], dim=0)
    return board_next, "continue"


def next_board_from_model_output(board_t, scores_t):
    board_next, _ = step_board_from_model_output(board_t, scores_t)
    return board_next


In [ ]:
def create_empty_board(board_size=BOARD_SIZE, device=None):
    board = torch.zeros(3, board_size, board_size, device=device)
    board[2, :, :] = 1
    return board


def create_empty_boards(batch_size, board_size=BOARD_SIZE, device=None):
    """
    Tạo batch gồm nhiều bàn cờ rỗng.

    Shape trả về là (B, 3, board_size, board_size), trong đó mỗi slot B là một ván riêng.
    Các ván này chạy song song nhưng có thể kết thúc ở thời điểm khác nhau.
    """
    boards = torch.zeros(batch_size, 3, board_size, board_size, device=device)
    boards[:, 2, :, :] = 1
    return boards


def clone_frozen_snapshot(model):
    """
    Tạo snapshot đóng băng của learner để làm đối thủ.

    Snapshot dùng cùng trọng số tại thời điểm bắt đầu train step, nhưng không nhận gradient.
    Điều này tránh việc learner đang update tự kéo luôn đối thủ thay đổi theo trong cùng graph.
    """
    snapshot = copy.deepcopy(model)
    snapshot.eval()
    for parameter in snapshot.parameters():
        parameter.requires_grad_(False)
    return snapshot


def learner_future_loss(
    local_losses,
    m=FUTURE_STEPS,
    gamma=GAMMA,
    future_weight=FUTURE_WEIGHT,
):
    """
    Gom loss cho một trajectory chỉ gồm các bước của learner trong cùng một ván.

    local_losses truyền vào hàm này phải thuộc cùng một game. Nếu đưa list phẳng của
    nhiều game vào đây, loss tương lai sẽ bị cộng nhầm từ game khác.
    """
    if local_losses.numel() == 0:
        return None

    num_moves = local_losses.shape[0]
    true_losses = []

    for t in range(num_moves):
        loss_t = local_losses[t]
        for i in range(1, m + 1):
            future_t = t + i
            if future_t >= num_moves:
                break
            loss_t = loss_t + future_weight * (gamma ** i) * local_losses[future_t]
        true_losses.append(loss_t)

    return torch.stack(true_losses).mean()


def learner_future_loss_by_game(
    local_losses,
    game_ids,
    batch_size,
    m=FUTURE_STEPS,
    gamma=GAMMA,
    future_weight=FUTURE_WEIGHT,
):
    """
    Tính future loss của learner nhưng tách riêng từng ván trong batch.

    rollout_batch_vs_snapshot_opponent lưu learner moves thành một list phẳng để dễ stack tensor.
    Vì các move trong list phẳng bị xen kẽ giữa nhiều game, ta phải dùng game_ids để gom lại:
      - lấy các local loss thuộc game 0 rồi tính learner_future_loss;
      - lấy các local loss thuộc game 1 rồi tính riêng;
      - cuối cùng average loss của các game có ít nhất một learner move.
    """
    per_game_losses = []

    for game_i in range(batch_size):
        mask = game_ids == game_i
        if not mask.any():
            continue

        game_loss = learner_future_loss(
            local_losses[mask],
            m=m,
            gamma=gamma,
            future_weight=future_weight,
        )
        if game_loss is not None:
            per_game_losses.append(game_loss)

    if len(per_game_losses) == 0:
        return None

    return torch.stack(per_game_losses).mean()


def rollout_batch_vs_snapshot_opponent(
    learner_model,
    snapshot_model,
    batch_size=TRAIN_BATCH_SIZE,
    max_moves=TRAIN_MAX_MOVES,
):
    """
    Chạy nhiều ván song song: learner đấu với snapshot frozen.

    Cơ chế batch fixed + done_mask:
      - boards luôn có shape (B, 3, 15, 15), không bị co lại khi có ván kết thúc sớm.
      - done[b] = True nghĩa là slot b đã xong, board slot đó đứng yên và không sinh thêm move.
      - active_mask = ~done chọn các ván còn đang chơi ở mỗi ply.

    Cơ chế train:
      - Chỉ các lượt của learner mới được lưu vào learner_boards/learner_scores.
      - Mỗi learner move cũng lưu learner_game_ids để future loss không cộng nhầm qua game khác.
      - Lượt snapshot chỉ dùng để cập nhật bàn cờ, chạy trong torch.no_grad(), không đóng góp loss.
    """
    device = next(learner_model.parameters()).device
    boards = create_empty_boards(batch_size=batch_size, board_size=BOARD_SIZE, device=device)

    # Nửa batch learner đi trước, nửa còn lại learner đi sau để học cả đen và trắng.
    learner_is_black = torch.arange(batch_size, device=device) % 2 == 0
    current_player_is_black = torch.ones(batch_size, dtype=torch.bool, device=device)

    done = torch.zeros(batch_size, dtype=torch.bool, device=device)
    terminal_reasons = ["max_moves" for _ in range(batch_size)]
    game_lengths = torch.zeros(batch_size, dtype=torch.long, device=device)

    learner_boards = []
    learner_scores = []
    learner_game_ids = []

    learner_wins = 0
    snapshot_wins = 0
    draws = 0
    invalid_games = 0

    learner_model.train()
    snapshot_model.eval()

    for move_number in range(1, max_moves + 1):
        active_indices = torch.nonzero(~done, as_tuple=False).flatten()
        if active_indices.numel() == 0:
            break

        # Một ván tới lượt learner khi màu hiện tại trùng với màu learner trong ván đó.
        learner_turn_mask = current_player_is_black[active_indices] == learner_is_black[active_indices]
        learner_indices = active_indices[learner_turn_mask]
        snapshot_indices = active_indices[~learner_turn_mask]

        if learner_indices.numel() > 0:
            active_boards = boards[learner_indices]
            active_scores = learner_model(active_boards)

            for local_i, game_i in enumerate(learner_indices.tolist()):
                board_before = boards[game_i]
                scores_t = active_scores[local_i]
                next_board, reason = step_board_from_model_output(board_before, scores_t)

                learner_boards.append(board_before.clone())
                learner_scores.append(scores_t)
                learner_game_ids.append(game_i)

                # Board state là dữ liệu môi trường, không phải tensor cần gradient.
                boards[game_i] = next_board.detach()
                game_lengths[game_i] = move_number

                if reason == "continue":
                    current_player_is_black[game_i] = ~current_player_is_black[game_i]
                else:
                    done[game_i] = True
                    terminal_reasons[game_i] = reason

                    if reason == "win":
                        learner_wins += 1
                    elif reason == "draw":
                        draws += 1
                    elif reason == "invalid":
                        invalid_games += 1

        if snapshot_indices.numel() > 0:
            with torch.no_grad():
                active_boards = boards[snapshot_indices]
                active_scores = snapshot_model(active_boards)

                for local_i, game_i in enumerate(snapshot_indices.tolist()):
                    board_before = boards[game_i]
                    scores_t = active_scores[local_i]
                    next_board, reason = step_board_from_model_output(board_before, scores_t)

                    boards[game_i] = next_board.detach()
                    game_lengths[game_i] = move_number

                    if reason == "continue":
                        current_player_is_black[game_i] = ~current_player_is_black[game_i]
                    else:
                        done[game_i] = True
                        terminal_reasons[game_i] = reason

                        if reason == "win":
                            snapshot_wins += 1
                        elif reason == "draw":
                            draws += 1
                        elif reason == "invalid":
                            invalid_games += 1

    if len(learner_boards) == 0:
        learner_boards_tensor = None
        learner_scores_tensor = None
        learner_game_ids_tensor = None
    else:
        learner_boards_tensor = torch.stack(learner_boards, dim=0)
        learner_scores_tensor = torch.stack(learner_scores, dim=0)
        learner_game_ids_tensor = torch.tensor(learner_game_ids, dtype=torch.long, device=device)

    unfinished_games = int((~done).sum().item())
    finished_games = batch_size - unfinished_games
    if unfinished_games > 0:
        for game_i in torch.nonzero(~done, as_tuple=False).flatten().tolist():
            terminal_reasons[game_i] = "max_moves"
            game_lengths[game_i] = max_moves

    return {
        "learner_boards": learner_boards_tensor,
        "learner_scores": learner_scores_tensor,
        "learner_game_ids": learner_game_ids_tensor,
        "terminal_reasons": terminal_reasons,
        "game_lengths": game_lengths,
        "finished_games": finished_games,
        "unfinished_games": unfinished_games,
        "learner_wins": learner_wins,
        "snapshot_wins": snapshot_wins,
        "draws": draws,
        "invalid_games": invalid_games,
        "learner_moves": len(learner_boards),
    }


def train_step_batch_vs_snapshot(
    model,
    optimizer,
    batch_size=TRAIN_BATCH_SIZE,
    max_moves=TRAIN_MAX_MOVES,
    m=FUTURE_STEPS,
    gamma=GAMMA,
    future_weight=FUTURE_WEIGHT,
):
    """
    1 bước train: learner đấu nhiều ván song song với snapshot frozen.

    Snapshot được tạo ở đầu step, nên trong toàn bộ rollout đối thủ là cố định.
    Optimizer chỉ update learner_model từ các nước learner đã lưu.
    """
    model.train()
    snapshot_model = clone_frozen_snapshot(model).to(next(model.parameters()).device)

    rollout = rollout_batch_vs_snapshot_opponent(
        learner_model=model,
        snapshot_model=snapshot_model,
        batch_size=batch_size,
        max_moves=max_moves,
    )

    if rollout["learner_boards"] is None:
        return None

    local_losses = compute_local_losses(rollout["learner_scores"], rollout["learner_boards"])
    if local_losses.numel() == 0:
        return None

    loss = learner_future_loss_by_game(
        local_losses,
        rollout["learner_game_ids"],
        batch_size=batch_size,
        m=m,
        gamma=gamma,
        future_weight=future_weight,
    )
    if loss is None:
        return None

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    game_lengths = rollout["game_lengths"].detach().float()

    return {
        "loss": float(loss.detach().item()),
        "avg_local_loss": float(local_losses.mean().detach().item()),
        "learner_moves": int(rollout["learner_moves"]),
        "finished_games": int(rollout["finished_games"]),
        "unfinished_games": int(rollout["unfinished_games"]),
        "learner_wins": int(rollout["learner_wins"]),
        "snapshot_wins": int(rollout["snapshot_wins"]),
        "draws": int(rollout["draws"]),
        "invalid_games": int(rollout["invalid_games"]),
        "avg_game_length": float(game_lengths.mean().item()),
    }


def train_loop_batch_vs_snapshot(
    model,
    optimizer,
    num_steps=TRAIN_STEPS,
    batch_size=TRAIN_BATCH_SIZE,
    max_moves=TRAIN_MAX_MOVES,
    m=FUTURE_STEPS,
    gamma=GAMMA,
    future_weight=FUTURE_WEIGHT,
    print_every=PRINT_EVERY,
):
    """
    Chạy train loop: batch nhiều ván song song, learner đấu snapshot frozen.
    """
    history = []

    for step in range(1, num_steps + 1):
        info = train_step_batch_vs_snapshot(
            model=model,
            optimizer=optimizer,
            batch_size=batch_size,
            max_moves=max_moves,
            m=m,
            gamma=gamma,
            future_weight=future_weight,
        )
        if info is None:
            continue

        history.append(info)

        if step % print_every == 0:
            print(
                f"step={step:03d} | loss={info['loss']:.4f} | "
                f"avg_local={info['avg_local_loss']:.4f} | learner_moves={info['learner_moves']} | "
                f"done={info['finished_games']}/{batch_size} | "
                f"W/L/D={info['learner_wins']}/{info['snapshot_wins']}/{info['draws']} | "
                f"unfinished={info['unfinished_games']} | avg_len={info['avg_game_length']:.1f}"
            )

    return history


In [ ]:
model = CaroResNet(input_channels=INPUT_CHANNELS, channels=MODEL_CHANNELS, num_blocks=MODEL_BLOCKS)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
history = train_loop_batch_vs_snapshot(
    model=model,
    optimizer=optimizer,
    num_steps=TRAIN_STEPS,
    batch_size=TRAIN_BATCH_SIZE,
    max_moves=TRAIN_MAX_MOVES,
    m=FUTURE_STEPS,
    gamma=GAMMA,
    future_weight=FUTURE_WEIGHT,
    print_every=PRINT_EVERY,
)

print("Số step train hợp lệ:", len(history))
if len(history) > 0:
    print("Loss cuối:", history[-1]["loss"])


In [ ]:
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display


def board_to_absolute_stones(board_t, current_player=1):
    """
    Đổi board từ góc nhìn tương đối sang quân đen/trắng tuyệt đối để vẽ.

    Trong lúc chơi, board_t[0] luôn là quân của người có lượt hiện tại.
    Nếu current_player == 1 thì người hiện tại là đen, ngược lại là trắng.
    """
    if current_player == 1:
        return board_t[0], board_t[1]
    return board_t[1], board_t[0]


def draw_board(board_t, move_number=0, current_player=1, reason="continue", pause=VISUAL_PAUSE):
    """
    Vẽ bàn cờ hiện tại.

    current_player phải khớp với góc nhìn của board_t. Với terminal board mới,
    step_board_from_model_output trả board theo góc nhìn người vừa đi, nên khi
    vẽ win/draw ta vẫn truyền current_player của người vừa đi để nước cuối hiện đúng màu.
    """
    black, white = board_to_absolute_stones(board_t, current_player=current_player)
    board_size = board_t.shape[-1]

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(6, 6))

    ax.set_xlim(-0.5, board_size - 0.5)
    ax.set_ylim(board_size - 0.5, -0.5)
    ax.set_xticks(range(board_size))
    ax.set_yticks(range(board_size))
    ax.grid(True)
    ax.set_aspect("equal")
    ax.set_title(f"Move {move_number} | turn={'black' if current_player == 1 else 'white'} | {reason}")

    black_pos = torch.nonzero(black > 0.5, as_tuple=False)
    white_pos = torch.nonzero(white > 0.5, as_tuple=False)

    if black_pos.numel() > 0:
        ax.scatter(black_pos[:, 1].cpu(), black_pos[:, 0].cpu(), s=180, c="black", marker="o")
    if white_pos.numel() > 0:
        ax.scatter(white_pos[:, 1].cpu(), white_pos[:, 0].cpu(), s=180, c="white", edgecolors="black", marker="o")

    display(fig)
    plt.close(fig)
    time.sleep(pause)


def play_two_models_visual(model_black, model_white=None, max_moves=VISUAL_MAX_MOVES, pause=VISUAL_PAUSE):
    """
    Cho 2 model chơi thử và hiển thị bàn cờ theo thời gian thực.

    Nếu model_white=None thì dùng cùng model cho cả 2 bên.
    Hàm này dùng step_board_from_model_output nên nước đi luôn được mask theo ô hợp lệ.
    """
    if model_white is None:
        model_white = model_black

    device = next(model_black.parameters()).device
    board_t = create_empty_board(board_size=BOARD_SIZE, device=device)
    current_player = 1
    reason = "continue"

    model_black.eval()
    model_white.eval()

    draw_board(board_t, move_number=0, current_player=current_player, reason=reason, pause=pause)

    with torch.no_grad():
        for move_number in range(1, max_moves + 1):
            model = model_black if current_player == 1 else model_white
            scores_t = model(board_t.unsqueeze(0)).squeeze(0)
            next_board, reason = step_board_from_model_output(board_t, scores_t)

            if reason == "continue":
                board_t = next_board
                current_player *= -1
                draw_board(board_t, move_number=move_number, current_player=current_player, reason=reason, pause=pause)
                continue

            # Với win/draw, next_board là board sau nước cuối từ góc nhìn người vừa đi.
            draw_board(next_board, move_number=move_number, current_player=current_player, reason=reason, pause=pause)
            print(f"Game ended: {reason} at move {move_number}")
            return reason, move_number

    print(f"Game ended: max_moves at move {max_moves}")
    return "max_moves", max_moves


# Chạy demo: nếu muốn chậm/nhanh hơn thì đổi VISUAL_PAUSE ở cell cấu hình.
play_two_models_visual(model)
